# 05 — Predict with the Trained Models

Uses the trained models saved in `MyDrive/FakeReviewProject/outputs/` (no retraining needed):

- `cnn_model.keras` + `cnn_vocab.json` + `cnn_scaler.pkl` + `cnn_config.json`
- `bert_model/` and `xlnet_model/` (weights, tokenizer, `predict_config.json`)

Any model that is not trained yet is skipped. A GPU is optional here: CPU works, it is just slower for BERT/XLNet.

In [ ]:
# --- Setup: mount Google Drive (dataset lives there, not in GitHub) ---
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = '/content/drive/MyDrive/FakeReviewProject'   # change if your folder differs
DATA_DIR    = f'{PROJECT_DIR}/dataset'                     # train.csv, test.csv
OUT_DIR     = f'{PROJECT_DIR}/outputs'
os.makedirs(OUT_DIR, exist_ok=True)
print(sorted(os.listdir(OUT_DIR)))

In [ ]:
!pip -q install nltk vaderSentiment
import json, pickle, re, numpy as np, pandas as pd, torch
from nltk.tokenize import TweetTokenizer
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
vader = SentimentIntensityAnalyzer()
MODELS = {}

# ---- CNN ----
if os.path.exists(f'{OUT_DIR}/cnn_model.keras'):
    import keras, tensorflow as tf
    cfg = json.load(open(f'{OUT_DIR}/cnn_config.json'))
    vocab = json.load(open(f'{OUT_DIR}/cnn_vocab.json'))
    scaler = pickle.load(open(f'{OUT_DIR}/cnn_scaler.pkl', 'rb'))
    # the custom HACE loss is only needed for training, so load without compiling
    cnn = keras.models.load_model(f'{OUT_DIR}/cnn_model.keras', compile=False)
    tok = TweetTokenizer(preserve_case=False, reduce_len=True, strip_handles=True)
    URL, HTML, NONTXT = re.compile(r'https?://\S+|www\.\S+'), re.compile(r'<[^>]+>'), re.compile(r"[^a-z0-9'!?.,:;()\s]")
    clean = lambda t: NONTXT.sub(' ', HTML.sub(' ', URL.sub(' ', str(t).lower())))

    def cnn_prob(texts, summaries, ratings):
        X = np.zeros((len(texts), cfg['max_len']), dtype='int32')
        for i, (t, s) in enumerate(zip(texts, summaries)):
            ids = [vocab.get(w, 1) for w in tok.tokenize(clean(s + ' . ' + t))][:cfg['max_len']]
            X[i, :len(ids)] = ids
        rs = [vader.polarity_scores(t)['compound'] for t in texts]
        feats = pd.DataFrame({'rating': ratings, 'review_length': [len(t.split()) for t in texts],
                              'review_sentiment': rs,
                              'summary_sentiment': [vader.polarity_scores(s)['compound'] for s in summaries],
                              'rating_sentiment_difference': np.asarray(ratings) / 5 - np.asarray(rs)})
        return cnn.predict([X, scaler.transform(feats[cfg['num_features']]).astype('float32')], verbose=0).ravel()
    MODELS['CNN'] = (cnn_prob, cfg['threshold'])

# ---- BERT / XLNet ----
from transformers import AutoTokenizer, AutoModelForSequenceClassification
device = 'cuda' if torch.cuda.is_available() else 'cpu'
for name in ['BERT', 'XLNet']:
    d = f'{OUT_DIR}/{name.lower()}_model'
    if not os.path.exists(d): continue
    tcfg = json.load(open(f'{d}/predict_config.json'))
    tk = AutoTokenizer.from_pretrained(d)
    mdl = AutoModelForSequenceClassification.from_pretrained(d).to(device).eval()
    def tf_prob(texts, summaries, ratings, tk=tk, mdl=mdl, L=tcfg['max_len']):
        out = []
        for i in range(0, len(texts), 64):
            enc = tk([s + ' . ' + t for t, s in zip(texts[i:i+64], summaries[i:i+64])], truncation=True,
                     max_length=L, padding=True, return_tensors='pt').to(device)
            with torch.no_grad(): out.append(torch.softmax(mdl(**enc).logits, -1)[:, 1].cpu().numpy())
        return np.concatenate(out)
    MODELS[name] = (tf_prob, tcfg['threshold'])

print('Loaded models:', list(MODELS))

In [ ]:
def predict(texts, summaries=None, ratings=None):
    """Return a table with each model's fake probability and FAKE/GENUINE label."""
    texts = [texts] if isinstance(texts, str) else list(texts)
    summaries = list(summaries) if summaries is not None else [''] * len(texts)
    ratings = list(ratings) if ratings is not None else [5.0] * len(texts)
    res = pd.DataFrame({'review': [t[:80] for t in texts]})
    for name, (fn, th) in MODELS.items():
        p = fn(texts, summaries, ratings)
        res[f'{name}_prob'] = p.round(4)
        res[name] = np.where(p >= th, 'FAKE', 'GENUINE')
    return res

## Try your own reviews

In [ ]:
predict(
    ['Absolutely amazing product!!! Best purchase ever, everyone should buy this, five stars!!!',
     'Case fits my phone well. The buttons are a bit stiff but it has survived two drops so far.'],
    summaries=['Amazing', 'Decent case'],
    ratings=[5, 4])

## Predict a whole CSV file
The CSV needs a `review_text` column; `review_summary` and `rating` are optional.

In [ ]:
CSV_IN = f'{PROJECT_DIR}/my_reviews.csv'   # change me
if os.path.exists(CSV_IN):
    df = pd.read_csv(CSV_IN)
    out = predict(df.review_text.astype(str),
                  df.get('review_summary', pd.Series([''] * len(df))).fillna('').astype(str),
                  df.get('rating', pd.Series([5.0] * len(df))).fillna(5.0))
    out.to_csv(f'{OUT_DIR}/my_reviews_predictions.csv', index=False)
    display(out.head())
else:
    print('Put a CSV at', CSV_IN)